## ___Data Subsetting___
-------------------

In [1]:
!python --version

Python 3.13.8


The system cannot find the path specified.


In [2]:
import numpy as np
import pandas as pd

In [3]:
fred = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Entire_Database_2021.csv", low_memory=False, header=0, skiprows=range(1, 10), encoding="latin1") # FRED v3
meta = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Column_Definitions_2021.csv", # usecols=["column_id", "name", "units"],
                                   index_col="column_id") # metadata for FRED v3 (column names, units etc.)

# https://github.com/traitecoevo/taxonlookup/releases/download/v1.1.5/plant_lookup.csv
# https://besjournals.onlinelibrary.wiley.com/doi/full/10.1111/2041-210X.12517
lookup = pd.read_csv(r"../../data/chapter2/plant_lookup.csv", low_memory=False, encoding="latin1", usecols=["genus", "apweb.family", # let's stick the the family info from APG website
                                                                                                         "order", "group"], index_col="genus").rename(mapper={"apweb.family": "family"}, axis=1) 

## ___Constants___
------------------

In [9]:
# CONSTANTS

COLLABORATION_GRADIENT_TRAITS = [
    "F00679", # RD
    "F00727", # SRL
    # "F00718", #	Specific root area (SRA)
    "F00104", # RCT
    "F00622", # Mycorrhiza_Fraction root length colonized by AM mycorrhizae
    "F00626", # Mycorrhiza_Fraction root length colonized by EM mycorrhizae
    "F00638", # Mycorrhiza_Fraction of root length
    "F00645"  # Mycorrhiza_Type_Data - not technically a root trait, just a relevant info
]

CONSERVATION_GRADIENT_TRAITS = [
    "F00709", # RTD
    "F00277", #	Root P content
    "F00261", # Root N content
    "F00358"  # Root lignin content
]

PLANT_TAXONOMY_ACCEPTED_COLUMNS = [
    "F01286", # Genus of plant according to The Plant List
    "F01287", # Species epithet of plant according to The Plant List
    "F01289", # Family of plant according to The Plant List
    "F01290"  # Order of plant. For Angiosperms, this was determined using the Angiosperm Phylogeny Website (APW): Stevens, P. F. (2001 onwards). Angiosperm Phylogeny Website. 
              # Version 14, July 2017 [and more or less continuously updated since]. Online at http://www.mobot.org/MOBOT/research/APweb/. 
              # For other plant groups (Gymnosperms, Pteridophytes, and Bryophytes) this was determined first using APW, but in cases where there were discrepancies within APW,
              # and to maintain consistency in suffix nomenclature (i.e. all orders ending in “iales”), additional sources were used including the USDA Plants Database 
              # (https://plants.usda.gov/) and the Missouri Botanical Garden (http://www.missouribotanicalgarden.org/plant-science/plant-science/research/plant-identification.aspx).
]

PLANT_TAXONOMY_DATA_SOURCE = [
    "F00017", # family
    "F00018", # genus
    "F00019"  # specific epithet
]

ESSENTIAL_COLUMNS = [
    "F00018", # genus
    "F00019", # species
    "F00056", # root order
    "F00059", # Notes_Root order classification scheme - Whether order scheme was centrifugal or centripetal. 
    # A centrifugal scheme counts the basal root as first order, with the distal root tip counted as the highest order (coarsest to finest). This is also known as the developmental approach.
    # A centripetal scheme counts the distal tip as the first order, with the basal root counted as the highest order (finest to coarsest). This is also known as the morphometric approach.
    "F00645",  # Mycorrhiza_Type_Data source - Type of mycorrhizae formed. 
    # AbtM = arbutoid mycorrhizae;
    # AM = arbuscular mycorrhizae;
    # DS = dark septate endophyte mycorrhizae;
    # EeM = ectendomycorrhizae;
    # EM = ectomycorrhizae;
    # ErM = ericoid mycorrhizae;
    # OrM = orchid mycorrhizae;
    # maherali_crude = means mycorrhizae are present but type is unknown;
    # NM = non-mycorrhizal.
    "F00043" # Plant photosynthetic pathway
]

BINOMINAL_NAME = [
    "F00018", # genus name
    "F00019", # specific epithet
    "F01286", # accepted genus name
    "F01287"  # accepted specific epithet
]

FRED_AND_TAXONLOOKUP_MERGED_TAXONOMY_COLUMNS = ["F01286", "F01287", "F01289", "F01290", "family", "order", "group"]

# based on the % of nans in the columns of RES traits

CHOSEN_RES_TRAITS = [
    "F00679", # RD
    "F00727", # SRL
    "F00261", # RN
    "F00709"  # RTD
]

CHOSEN_RES_TRAITS_NAMES = ["RD", "SRL", "RN", "RTD"]

In [6]:
meta.loc["F00261", :]

name                                                 Root N content
units                                                          mg/g
definition        Mass of nitrogen per root mass for sampled roots.
type                                                    Root_Traits
data_type                                                       NaN
category                                                  Chemistry
trait_category                                       Macronutrients
trait_type                                           Root N content
root_trait                                                     True
sort_order                                                      209
count                                                          6764
count_values      12 (41)\n8 (36)\n9 (33)\n10 (32)\n7 (30)\n10.1...
Name: F00261, dtype: object

## ___Exploratory Analyses___
-----------------------

In [10]:
# subset the dataframe to pick only the columns with least amount of missing values
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS]

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
0,NaN,NaN,NaN,NaN,Dicranopteris,dichotoma,1.0,centripetal,AM,C3,Dicranopteris,linearis,Gleicheniaceae,Gleicheniales
1,NaN,NaN,NaN,NaN,Dicranopteris,dichotoma,2.0,centripetal,AM,C3,Dicranopteris,linearis,Gleicheniaceae,Gleicheniales
2,NaN,NaN,NaN,NaN,Dicranopteris,dichotoma,3.0,centripetal,AM,C3,Dicranopteris,linearis,Gleicheniaceae,Gleicheniales
3,NaN,NaN,NaN,NaN,Cunninghamia,lanceolata,1.0,centripetal,AM,C3,Cunninghamia,lanceolata,Cupressaceae,Pinales
4,NaN,NaN,NaN,NaN,Cunninghamia,lanceolata,2.0,centripetal,AM,C3,Cunninghamia,lanceolata,Cupressaceae,Pinales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57185,NaN,0.567694,8.666348,NaN,Diplopterygium,chinense,NaN,NaN,NaN,NaN,Diplopterygium,chinensis,Gleicheniaceae,Gleicheniales
57186,NaN,0.693820,8.520457,NaN,Diplopterygium,chinense,NaN,NaN,NaN,NaN,Diplopterygium,chinensis,Gleicheniaceae,Gleicheniales
57187,NaN,3.364970,14.188312,NaN,Osmunda,japonica,NaN,NaN,NaN,NaN,Osmunda,japonica,Osmundaceae,Osmundaceae
57188,NaN,2.174190,14.849199,NaN,Osmunda,japonica,NaN,NaN,NaN,NaN,Osmunda,japonica,Osmundaceae,Osmundaceae


In [11]:
# get rid of records with missing values for the 4 chosen root traits
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS)

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
302,0.250000,71.600000,19.700000,0.440000,NaN,NaN,1.0,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
303,0.290000,45.900000,14.900000,0.300000,NaN,NaN,1.0,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
304,0.380000,68.100000,20.400000,0.320000,NaN,NaN,1.0,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
382,0.142951,447.167648,18.478265,0.169294,Populus,trichocarpa,1.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
383,0.249630,166.388374,14.456530,0.160036,Populus,trichocarpa,2.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54941,0.400000,51.410000,16.500000,0.150000,Allium,tenuissimum,NaN,centripetal,NaN,C3,Allium,tenuissimum,Amaryllidaceae,Asparagales
54942,0.490000,31.670000,16.500000,0.170000,Allium,bidentatum,NaN,centripetal,NaN,C3,Allium,bidentatum,Amaryllidaceae,Asparagales
54943,0.510000,26.060000,16.500000,0.190000,Allium,bidentatum,NaN,centripetal,NaN,C3,Allium,bidentatum,Amaryllidaceae,Asparagales
54944,0.230000,102.900000,12.200000,0.230000,Chamaerhodos,erecta,NaN,centripetal,NaN,C3,Chamaerhodos,erecta,Rosaceae,Rosales


In [13]:
# see whether accepted genus source has info where the data source genus name is missing!!
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS).query("F00018.isna()") # NOPE

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
302,0.250000,71.600000,19.700000,0.44,NaN,NaN,1.0,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
303,0.290000,45.900000,14.900000,0.30,NaN,NaN,1.0,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
304,0.380000,68.100000,20.400000,0.32,NaN,NaN,1.0,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
1603,0.510000,30.160000,12.400000,0.27,NaN,NaN,NaN,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
1604,0.490000,35.250000,8.200000,0.28,NaN,NaN,NaN,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
1605,0.510000,32.550000,9.700000,0.27,NaN,NaN,NaN,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
1606,0.510000,31.770000,9.100000,0.27,NaN,NaN,NaN,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
1607,0.490000,34.440000,8.300000,0.29,NaN,NaN,NaN,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
1608,0.490000,27.630000,7.000000,0.32,NaN,NaN,NaN,centripetal,NaN,NaN,NaN,NaN,NaN,NaN
1609,0.490000,33.500000,7.100000,0.33,NaN,NaN,NaN,centripetal,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# drop the rows that do not have a genus data source (F00018)
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + ["F00018"])

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
382,0.142951,447.167648,18.478265,0.169294,Populus,trichocarpa,1.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
383,0.249630,166.388374,14.456530,0.160036,Populus,trichocarpa,2.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
384,0.425085,60.723679,10.760881,0.171351,Populus,trichocarpa,3.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
385,0.687417,30.725767,9.130448,0.191318,Populus,trichocarpa,4.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
386,0.175513,330.424588,22.065220,0.169294,Populus,tremula,1.0,centripetal,NaN,C3,Populus,tremula,Salicaceae,Malpighiales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54941,0.400000,51.410000,16.500000,0.150000,Allium,tenuissimum,NaN,centripetal,NaN,C3,Allium,tenuissimum,Amaryllidaceae,Asparagales
54942,0.490000,31.670000,16.500000,0.170000,Allium,bidentatum,NaN,centripetal,NaN,C3,Allium,bidentatum,Amaryllidaceae,Asparagales
54943,0.510000,26.060000,16.500000,0.190000,Allium,bidentatum,NaN,centripetal,NaN,C3,Allium,bidentatum,Amaryllidaceae,Asparagales
54944,0.230000,102.900000,12.200000,0.230000,Chamaerhodos,erecta,NaN,centripetal,NaN,C3,Chamaerhodos,erecta,Rosaceae,Rosales


In [15]:
# missing values???
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + ["F00018"]).isna().mean()

F00679    0.000000
F00727    0.000000
F00261    0.000000
F00709    0.000000
F00018    0.000000
F00019    0.009738
F00056    0.435206
F00059    0.318352
F00645    0.501124
F00043    0.221723
F01286    0.000749
F01287    0.009738
F01289    0.014981
F01290    0.014981
dtype: float64

In [17]:
# root order, photosynthetic pathway, root order method info and mycorrhizal type still have lots of NAs

In [16]:
# see how the root order was determined
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + ["F00018"]).F00059.value_counts(dropna=False) # all centripetal, we're good :)

F00059
centripetal    910
NaN            425
Name: count, dtype: int64

In [18]:
# there are rows where the root order method info is present but the actual root order is missing???
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + ["F00018"]).loc[:, ["F00056", "F00059"]].query("not F00059.isna() and F00056.isna()")
# 190 rows???

,F00056,F00059
1553,NaN,centripetal
1554,NaN,centripetal
1555,NaN,centripetal
1556,NaN,centripetal
1557,NaN,centripetal
...,...,...
54941,NaN,centripetal
54942,NaN,centripetal
54943,NaN,centripetal
54944,NaN,centripetal


In [19]:
# including root order (F00056) in NA removal columns
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + ["F00018", "F00056"])

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
382,0.142951,447.167648,18.478265,0.169294,Populus,trichocarpa,1.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
383,0.249630,166.388374,14.456530,0.160036,Populus,trichocarpa,2.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
384,0.425085,60.723679,10.760881,0.171351,Populus,trichocarpa,3.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
385,0.687417,30.725767,9.130448,0.191318,Populus,trichocarpa,4.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
386,0.175513,330.424588,22.065220,0.169294,Populus,tremula,1.0,centripetal,NaN,C3,Populus,tremula,Salicaceae,Malpighiales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45575,0.042154,81.443333,19.366667,0.103333,Prunus,sargentii,1.0,NaN,NaN,C3,Prunus,sargentii,Rosaceae,Rosales
45576,0.219538,20.513333,23.200000,0.116667,Sassafras,albidum,1.0,NaN,NaN,C3,Sassafras,albidum,Lauraceae,Laurales
45577,0.086988,36.817500,26.300000,0.155000,Styphnolobium,japonicum,1.0,NaN,NaN,C3,Styphnolobium,japonicum,Fabaceae,Fabales
45578,0.061422,89.920000,23.900000,0.075000,Syringa,reticulata,1.0,NaN,NaN,C3,Syringa,reticulata,Oleaceae,Lamiales


In [20]:
# including mycorrhizal type (F00645) in NA removal columns
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + ["F00018", "F00645"])

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
1553,0.3100,23.800000,13.01,0.550000,Aristotelia,serrata,NaN,centripetal,AM,C3,Aristotelia,serrata,Elaeocarpaceae,Oxalidales
1554,0.5300,22.200000,9.01,0.200000,Ascarina,lucida,NaN,centripetal,AM,NaN,Ascarina,lucida,Chloranthaceae,Chloranthales
1555,0.4300,31.800000,16.18,0.200000,Asplenium,bulbiferum,NaN,centripetal,AM,C3,Asplenium,bulbiferum,Aspleniaceae,Polypodiales
1556,0.9400,12.900000,10.12,0.090000,Astelia,fragrans,NaN,centripetal,AM,C3,Astelia,fragrans,Asteliaceae,Asparagales
1557,0.9500,14.500000,6.49,0.090000,Astelia,nervosa,NaN,centripetal,AM,C3,Astelia,nervosa,Asteliaceae,Asparagales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42456,0.3641,55.618299,9.80,0.172743,Weinmannia,silvicola,NaN,NaN,AM,NaN,Weinmannia,silvicola,Cunoniaceae,Oxalidales
42457,0.3474,65.650773,9.80,0.160465,Weinmannia,silvicola,NaN,NaN,AM,NaN,Weinmannia,silvicola,Cunoniaceae,Oxalidales
42458,0.4144,38.274731,9.80,0.193932,Weinmannia,silvicola,NaN,NaN,AM,NaN,Weinmannia,silvicola,Cunoniaceae,Oxalidales
42459,0.3619,61.581617,9.80,0.157944,Weinmannia,silvicola,NaN,NaN,AM,NaN,Weinmannia,silvicola,Cunoniaceae,Oxalidales


In [21]:
# including root order and mycorrhizal type in NA removal columns
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + ["F00018", "F00056", "F00645"])

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
7575,0.373683,32.822980,17.884615,0.370787,Fraxinus,excelsior,1.0,centripetal,AM,C3,Fraxinus,excelsior,Oleaceae,Lamiales
7578,0.477964,16.411514,15.576925,0.286517,Fraxinus,excelsior,2.0,centripetal,AM,C3,Fraxinus,excelsior,Oleaceae,Lamiales
7581,0.704767,6.028751,13.461542,0.320225,Fraxinus,excelsior,3.0,centripetal,AM,C3,Fraxinus,excelsior,Oleaceae,Lamiales
7584,1.084639,3.684259,12.211543,0.353933,Fraxinus,excelsior,4.0,centripetal,AM,C3,Fraxinus,excelsior,Oleaceae,Lamiales
7587,0.266508,62.631585,14.230772,0.380899,Acer,pseudoplatanus,1.0,centripetal,AM,C3,Acer,pseudoplatanus,Sapindaceae,Sapindales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24549,0.283178,55.472897,17.700000,0.286228,Acer,truncatum,2.0,centripetal,AM + EM,C3,Acer,truncatum,Sapindaceae,Sapindales
24550,0.345455,48.454545,15.000000,0.220188,Acer,truncatum,3.0,centripetal,AM + EM,C3,Acer,truncatum,Sapindaceae,Sapindales
24551,0.471667,16.750000,12.300000,0.341684,Acer,truncatum,4.0,centripetal,AM + EM,C3,Acer,truncatum,Sapindaceae,Sapindales
24552,0.782500,7.882690,11.100000,0.263795,Acer,truncatum,5.0,centripetal,AM + EM,C3,Acer,truncatum,Sapindaceae,Sapindales


In [22]:
# if available in literature, mycorrhizal type, photosynthetic pathway can be manually added to the data, so do not prioritize those now
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + ["F00018"]).query("F00056.isin((1, 2, 3))")

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
382,0.142951,447.167648,18.478265,0.169294,Populus,trichocarpa,1.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
383,0.249630,166.388374,14.456530,0.160036,Populus,trichocarpa,2.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
384,0.425085,60.723679,10.760881,0.171351,Populus,trichocarpa,3.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
386,0.175513,330.424588,22.065220,0.169294,Populus,tremula,1.0,centripetal,NaN,C3,Populus,tremula,Salicaceae,Malpighiales
387,0.285828,116.664475,16.304355,0.160041,Populus,tremula,2.0,centripetal,NaN,C3,Populus,tremula,Salicaceae,Malpighiales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45575,0.042154,81.443333,19.366667,0.103333,Prunus,sargentii,1.0,NaN,NaN,C3,Prunus,sargentii,Rosaceae,Rosales
45576,0.219538,20.513333,23.200000,0.116667,Sassafras,albidum,1.0,NaN,NaN,C3,Sassafras,albidum,Lauraceae,Laurales
45577,0.086988,36.817500,26.300000,0.155000,Styphnolobium,japonicum,1.0,NaN,NaN,C3,Styphnolobium,japonicum,Fabaceae,Fabales
45578,0.061422,89.920000,23.900000,0.075000,Syringa,reticulata,1.0,NaN,NaN,C3,Syringa,reticulata,Oleaceae,Lamiales


In [24]:
# missing values???
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + ["F00018"]).query("F00056.isin((1, 2, 3))").isna().mean()

F00679    0.000000
F00727    0.000000
F00261    0.000000
F00709    0.000000
F00018    0.000000
F00019    0.000000
F00056    0.000000
F00059    0.056385
F00645    0.371476
F00043    0.145937
F01286    0.000000
F01287    0.000000
F01289    0.000000
F01290    0.000000
dtype: float64

In [25]:
# why not consider specific epithet for the NA removal???
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + BINOMINAL_NAME).query("F00056.isin((1, 2, 3))")

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
382,0.142951,447.167648,18.478265,0.169294,Populus,trichocarpa,1.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
383,0.249630,166.388374,14.456530,0.160036,Populus,trichocarpa,2.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
384,0.425085,60.723679,10.760881,0.171351,Populus,trichocarpa,3.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
386,0.175513,330.424588,22.065220,0.169294,Populus,tremula,1.0,centripetal,NaN,C3,Populus,tremula,Salicaceae,Malpighiales
387,0.285828,116.664475,16.304355,0.160041,Populus,tremula,2.0,centripetal,NaN,C3,Populus,tremula,Salicaceae,Malpighiales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45575,0.042154,81.443333,19.366667,0.103333,Prunus,sargentii,1.0,NaN,NaN,C3,Prunus,sargentii,Rosaceae,Rosales
45576,0.219538,20.513333,23.200000,0.116667,Sassafras,albidum,1.0,NaN,NaN,C3,Sassafras,albidum,Lauraceae,Laurales
45577,0.086988,36.817500,26.300000,0.155000,Styphnolobium,japonicum,1.0,NaN,NaN,C3,Styphnolobium,japonicum,Fabaceae,Fabales
45578,0.061422,89.920000,23.900000,0.075000,Syringa,reticulata,1.0,NaN,NaN,C3,Syringa,reticulata,Oleaceae,Lamiales


In [26]:
# why not consider both data source taxonomy and accepted taxonomy??
fred.loc[:, CHOSEN_RES_TRAITS + ESSENTIAL_COLUMNS + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + BINOMINAL_NAME + PLANT_TAXONOMY_ACCEPTED_COLUMNS).query("F00056.isin((1, 2, 3))")

,F00679,F00727,F00261,F00709,F00018,F00019,F00056,F00059,F00645,F00043,F01286,F01287,F01289,F01290
382,0.142951,447.167648,18.478265,0.169294,Populus,trichocarpa,1.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
383,0.249630,166.388374,14.456530,0.160036,Populus,trichocarpa,2.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
384,0.425085,60.723679,10.760881,0.171351,Populus,trichocarpa,3.0,centripetal,NaN,C3,Populus,trichocarpa,Salicaceae,Malpighiales
386,0.175513,330.424588,22.065220,0.169294,Populus,tremula,1.0,centripetal,NaN,C3,Populus,tremula,Salicaceae,Malpighiales
387,0.285828,116.664475,16.304355,0.160041,Populus,tremula,2.0,centripetal,NaN,C3,Populus,tremula,Salicaceae,Malpighiales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45575,0.042154,81.443333,19.366667,0.103333,Prunus,sargentii,1.0,NaN,NaN,C3,Prunus,sargentii,Rosaceae,Rosales
45576,0.219538,20.513333,23.200000,0.116667,Sassafras,albidum,1.0,NaN,NaN,C3,Sassafras,albidum,Lauraceae,Laurales
45577,0.086988,36.817500,26.300000,0.155000,Styphnolobium,japonicum,1.0,NaN,NaN,C3,Styphnolobium,japonicum,Fabaceae,Fabales
45578,0.061422,89.920000,23.900000,0.075000,Syringa,reticulata,1.0,NaN,NaN,C3,Syringa,reticulata,Oleaceae,Lamiales


## ___Subsetting___
-------------------

In [30]:
# FINE ABSORPTIVE ROOTS

# in our subset fine roots means roots of orders 1, 2 or 3 
# OR roots with diameter less than 1 mm (being more stringent here because the order is unknown) - TURNED OUT TO BE A BAD IDEA
# F00004 - Data source_Citation

data = fred.loc[:, CHOSEN_RES_TRAITS + ["F00043", "F00056", "F00645", "F00004"] + PLANT_TAXONOMY_ACCEPTED_COLUMNS].dropna(subset=CHOSEN_RES_TRAITS + PLANT_TAXONOMY_ACCEPTED_COLUMNS).\
                query("F00056.isin((1, 2, 3))").reset_index(drop=True)
subset_unique_genera = data.F01286.unique() # all the unique genus names in the subset

## ___Taxonomic Corrections___
--------------------

In [31]:
lookup.head()

,family,order,group
genus,,,
Acorus,Acoraceae,Acorales,Angiosperms
Albidella,Alismataceae,Alismatales,Angiosperms
Alisma,Alismataceae,Alismatales,Angiosperms
Astonia,Alismataceae,Alismatales,Angiosperms
Baldellia,Alismataceae,Alismatales,Angiosperms


In [32]:
# family level conflicts, whether a genus is classified under multiple families
np.array([data.query("F01286==@genus").F01289.unique().size for genus in subset_unique_genera]).max()

np.int64(2)

In [33]:
# where???
np.where(np.array([data.query("F01286==@genus").F01289.unique().size for genus in subset_unique_genera]) > 1)

(array([26]),)

In [34]:
# order level conflicts, whether a genus is classified under multiple orders
np.array([data.query("F01286==@genus").F01290.unique().size for genus in subset_unique_genera]).max()

np.int64(2)

In [35]:
# where???
np.where(np.array([data.query("F01286==@genus").F01290.unique().size for genus in subset_unique_genera]) > 1)

(array([26]),)

In [36]:
# what's the 26th record???
subset_unique_genera[np.where(np.array([data.query("F01286==@genus").F01290.unique().size for genus in subset_unique_genera]) > 1)]

array(['Platanus'], dtype=object)

In [37]:
data.query("F01286==\"Platanus\"") # Platanaceae and Sapindaceae

,F00679,F00727,F00261,F00709,F00043,F00056,F00645,F00004,F01286,F01287,F01289,F01290
126,0.486750,11.977390,9.219875,0.1490,C3,1.0,AM,"Valverde-Barrantes OJ, Smemo KA, Blackwood CB...",Platanus,occidentalis,Platanaceae,Proteales
127,1.169600,3.218217,6.524208,0.3000,C3,2.0,AM,"Valverde-Barrantes OJ, Smemo KA, Blackwood CB...",Platanus,occidentalis,Platanaceae,Proteales
128,0.416000,23.929697,13.021250,0.1500,C3,3.0,AM,"Valverde-Barrantes OJ, Smemo KA, Blackwood CB...",Platanus,occidentalis,Platanaceae,Proteales
596,0.282061,23.930000,13.000000,0.1475,C3,1.0,NaN,Valverde et al (unpublished),Platanus,acerifolia,Sapindaceae,Sapindales


In [38]:
# the correct family for genus Platanus is Platanaceae
lookup.loc["Platanus"]

family    Platanaceae
order       Proteales
group     Angiosperms
Name: Platanus, dtype: object

In [39]:
idx_platanus = data.query("F01286==\"Platanus\" and F01289==\"Sapindaceae\" and F01290==\"Sapindales\"").index
# fix it
data.loc[idx_platanus, "F01289"] = "Platanaceae"
data.loc[idx_platanus, "F01290"] = "Proteales"

In [40]:
# try looking up using the accepted genus name instead of the data source???
lookup.loc[data.F01286] # works right :)

,family,order,group
genus,,,
Populus,Salicaceae,Malpighiales,Angiosperms
Populus,Salicaceae,Malpighiales,Angiosperms
Populus,Salicaceae,Malpighiales,Angiosperms
Populus,Salicaceae,Malpighiales,Angiosperms
Populus,Salicaceae,Malpighiales,Angiosperms
...,...,...,...
Prunus,Rosaceae,Rosales,Angiosperms
Sassafras,Lauraceae,Laurales,Angiosperms
Styphnolobium,Fabaceae,Fabales,Angiosperms


In [41]:
combined_taxonomy = pd.merge(left=data, right=lookup, left_on="F01286", right_index=True) # lookup's index is genus names so;
combined_taxonomy.head()

,F00679,F00727,F00261,F00709,F00043,F00056,F00645,F00004,F01286,F01287,F01289,F01290,family,order,group
0,0.142951,447.167648,18.478265,0.169294,C3,1.0,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Populus,trichocarpa,Salicaceae,Malpighiales,Salicaceae,Malpighiales,Angiosperms
1,0.249630,166.388374,14.456530,0.160036,C3,2.0,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Populus,trichocarpa,Salicaceae,Malpighiales,Salicaceae,Malpighiales,Angiosperms
2,0.425085,60.723679,10.760881,0.171351,C3,3.0,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Populus,trichocarpa,Salicaceae,Malpighiales,Salicaceae,Malpighiales,Angiosperms
3,0.175513,330.424588,22.065220,0.169294,C3,1.0,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Populus,tremula,Salicaceae,Malpighiales,Salicaceae,Malpighiales,Angiosperms
4,0.285828,116.664475,16.304355,0.160041,C3,2.0,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Populus,tremula,Salicaceae,Malpighiales,Salicaceae,Malpighiales,Angiosperms


In [42]:
# family level conflicts
combined_taxonomy.query("F01289!=family").loc[:, FRED_AND_TAXONLOOKUP_MERGED_TAXONOMY_COLUMNS].drop_duplicates()

,F01286,F01287,F01289,F01290,family,order,group


In [43]:
# order level conflicts
# the order information in FRED v3 seems more up to date compared to the info in the lookup table! yikes!
combined_taxonomy.query("F01290!=order").loc[:, FRED_AND_TAXONLOOKUP_MERGED_TAXONOMY_COLUMNS].drop_duplicates().sort_values(by=PLANT_TAXONOMY_ACCEPTED_COLUMNS)#.shape

,F01286,F01287,F01289,F01290,family,order,group
249,Athyrium,multidentatum,Athyriaceae,Polypodiales,Athyriaceae,Eupolypod II,Pteridophytes
252,Athyrium,sinense,Athyriaceae,Polypodiales,Athyriaceae,Eupolypod II,Pteridophytes
270,Athyrium,spinulosum,Athyriaceae,Polypodiales,Athyriaceae,Eupolypod II,Pteridophytes
255,Cystopteris,sudetica,Cystopteridaceae,Polypodiales,Cystopteridaceae,Polypodiales-Eupolypod II,Pteridophytes
264,Matteuccia,struthiopteris,Onocleaceae,Polypodiales,Onocleaceae,Eupolypod II,Pteridophytes
267,Onoclea,sensibilis,Onocleaceae,Polypodiales,Onocleaceae,Eupolypod II,Pteridophytes


In [44]:
# safety assertions before creating the categorical subset
# no family level misclassifications
assert (np.array([data.query(r"F01286==@g").F01289.unique().size for g in subset_unique_genera]) == 1).all()
# no order level misclassifications
assert (np.array([data.query(r"F01286==@g").F01290.unique().size for g in subset_unique_genera]) == 1).all()

In [45]:
# the same subset, for categorical data exploration - mycorrhizal states and photosynthetic pathways
data_categorical = data.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + ["F00043", "F00645", "F00004"]].drop_duplicates(subset=PLANT_TAXONOMY_ACCEPTED_COLUMNS + ["F00043",
                                    "F00645"]).reset_index(drop=True) # do not consider the citations column for duplicates
data_categorical.insert(loc=0, column="binominal", value=data_categorical.F01286 + ' ' + data_categorical.F01287) # does an in place insert, concatenating the genus and specific epithets

In [50]:
data_taxa = data.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS].drop_duplicates()

In [52]:
# serialize the dataframes to .csv files
data.to_csv(r"../../data/chapter2/FREDv3subset/FRED_subset.csv", index=False)
data_categorical.to_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_categorical.csv", index=False)
data_taxa.to_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_unique_taxa.csv", index=False)

In [53]:
# genus - family relationships
# follow the format of https://raw.githubusercontent.com/megatrees/amphibian_20221117/main/amphibian_genus_list.csv
data_taxa.loc[:, ["F01286", "F01289"]].drop_duplicates().to_csv(r"../../data/chapter2/FREDv3subset/fred_gen_fam.csv", index=False)

In [54]:
# binominal name, genus name followed by two empty spaces
# follows the format of https://raw.githubusercontent.com/megatrees/amphibian_20221117/main/amphibian_sample_species_list.csv
pd.DataFrame({"species": data_taxa.F01286 + ' ' + data_taxa.F01287, "genus": data_taxa.F01286, "family": None, "species.relative": None, "genus.relative": None}).\
                        drop_duplicates().reset_index(drop=True).to_csv(r"../../data/chapter2/FREDv3subset/fred_binom_genus.csv", index=False)